#**Exercise 3: Query Metadata and Data from HydroServer using hydroserverpy**

### **Overview**

This Jupyter Notebook demonstrates how to query time-series data and metadata from a datastream stored in HydroServer using the hydroserverpy Python package. The data are returned as a Pandas DataFrame, making them easy to manipulate, analyze, and visualize.

More detailed examples of how to use [hydroserverpy](https://pypi.org/project/hydroserverpy/) are available in HydroServer's documentation at:

* https://hydroserver2.github.io/hydroserver/user-guides/how-to/using-the-python-client.html
* https://www.hydroserver.org

### **Prerequisites**

You must have an account on the HydroServer Playground instance to run this notebook. If you haven't set up your user account yet, navigate to https://playground.hydroserver.org and follow the instructions to create a new user account.

### **References**

Code adapted by Sara Alonso Vicario from the Center for Geospatial Solutions from examples developed by the HydroServer development team, including Jeff Horsburgh, Ken Lippold, and Daniel Slaugh. The original materials are available in this [HydroShare resource](https://www.hydroshare.org/resource/136c2bc6512540d59ce707bbd9c93e8a/).


## 1. **Getting Started**

---

### **Install hydroserverpy**

For this workshop, we will use Google Colab to run the exercises. Before starting, run the code cell below to install the required version of the hydroserverpy package. For this exercise, we will connect to the HydroServer Playground instance at playground.hydroserver.org. The current version of hydroserverpy used for this training is [1.11.3.](https://pypi.org/project/hydroserverpy/)

In [ ]:
!pip install hydroserverpy==1.11.3

### **Import the Required Python Packages**

First, we import the Python packages needed for this exercise:

- **hydroserverpy** – Connects to HydroServer and allows us to query resources, metadata, and observations.
- **datetime** – Helps define and work with dates and times.
- **matplotlib** – Used to visualize the observations retrieved from HydroServer.
- **getpass** – Allows you to enter your HydroServer password securely without displaying it on the screen.

In [ ]:
from hydroserverpy import HydroServer
from datetime import datetime
import matplotlib.pyplot as plt
from getpass import getpass

### **Set the Initial Parameters to Connect to HydroServer**

The first step in interacting with a HydroServer instance is to create a connection to that instance. For this example, we will use your username and password because we will create the Workspace in code.

**IMPORTANT: In the following code, change the email to match the HydroServer user account you created.**

In [ ]:
# Set initial parameters to connect to HydroServer
hydroserver_host = 'https://playground.hydroserver.org'

# Change the email and password below to your HydroServer username and password
hydroserver_email = 'your email' #'user@youremail.com'
hydroserver_password = getpass('Enter your HydroServer password: ') #getpass('Enter your HydroServer password: ')

### **Initialize HydroServer Connection**

Initialize the connection to HydroServer with the connection information specified above.

In [ ]:
# Initialize HydroServer connection with credentials.
hs = HydroServer(
    host=hydroserver_host,
    email=hydroserver_email,
    password=hydroserver_password
)

print('\nSuccessfully connected to HydroServer!')

## **2. Querying Metadata from HydroServer**

---

Querying metadata from HydroServer is pretty straightforward once you have a basic understanding of HydroServer's data model.

We'll start with getting information about Workspaces. Generally, a Workspace is a container within which you organize monitoring sites (Things).

Each monitoring site can have any number of Datastreams. A Datastream is a time series of Observation values. There's other important metadata, but the important heirarchy in Hydroserver is:

Workspace --> Thing --> Datastream --> Observations

### **Query Workspaces**

We'll start by getting a list of Workspaces I have access to. These are workspaces that I own, have been shared with me, or that are publicly available.

In [ ]:
workspaces = hs.workspaces.list()

# Print the names of the first 5 workspaces that were returned:
print(f'\nList of workspaces I have access to:\n')
for workspace in workspaces.items[:5]:
    print(workspace.name)

### **Query Workspaces You are Affiliated With**

Many of the workspaces on the Playground instance of HydroServer were set up for testing and aren't relevant to you.

So, let's limit the query to return only workspaces with which you are affiliated (you either own them or you have been given access).

In [ ]:
# Limit the list to workspaces I am affiliated with.
my_workspaces = hs.workspaces.list(is_associated=True)

print(f'\nList of workspaces I am affiliated with:\n')
for workspace in my_workspaces.items:
    print(workspace.name)

### **Query a Workspace with a Specific Name**

To get a specific workspace by name, first retrieve your list of workspaces, then search the list for the workspace with the name you want.

**IMPORTANT: Replace the workspace name below with the name of your own workspace.**

In [ ]:
workspace = [
    workspace
    for workspace in my_workspaces.items
    if workspace.name == "Uganda Training 2026 - your name"  # Change this name to your workspace name
][0]

workspace_id = workspace.uid

print(f"\nRetrieved workspace: {workspace.name}")
print(f"\nRetrieved workspace uid: {workspace_id}")

### **Query Monitoring Sites (Things) in my Workspace**

We can also query information about monitoring sites (Things) in my workspace.

The Workspace used here is the Workspace returned in the last step where we searched for a Workspace with a specific name.

In [ ]:
# Limit the list to the Things in a particular workspace
workspace_things = workspace.things
for thing in workspace_things:
    print(thing.name)

### **Query a Monitoring Site with a Specific Name**

To retrieve a specific Thing by name, first get the list of Things in the Workspace and then find the Thing you want.

The code below first finds the Thing by name and gets its UUID.

**IMPORTANT: If you want to search for a Thing with a different name, change the name of the Thing on the following block code.**

In [ ]:
thing = [
    thing
    for thing in workspace_things
    if thing.name == "Kanzenze Hydrological Station"  # Set the name of the thing you want to find
][0]

# Get a thing using its identifier
thing_id = thing.uid  # Get the ID for a thing

print(f"\nRetrieved thing: {thing.name}")

### **Query Attributes of a Monitoring Site (Thing)**

The metadata you get back from HydroServer is organized in Python objects that can easily be queried.

The following code shows how to extract some the attributes for the Thing object you just retrieved in the last step.

In [ ]:
# Get attributes of a Thing
print(f'\nExtracting attributes of a Thing:\n')
print(f'Site name: {thing.name}')
print(f'Site description: {thing.description}')
print(f'Latitude: {thing.latitude}')
print(f'Longitude: {thing.longitude}')
print(f'Site type: {thing.site_type}')

# You can get the same information if you know the UUID for a Thing
#thing = hs.things.get(uid=thing.uid)  # Get the thing that goes with that ID
#print(f'\nExtracting attributes of a Thing using its ID:\n')
#print(f'Site name: {thing.name}')
#print(f'Site description: {thing.description}')
#print(f'Latitude: {thing.latitude}')
#print(f'Longitude: {thing.longitude}')
#print(f'Site type: {thing.site_type}')

### **Additional Examples of Querying Monitoring Site (Thing) Metadata**

The code below shows how to retrieve all the **attributes** containing the metadata for a **monitoring site (Thing)**.

In [ ]:
# Basic information
print(f"UID: {thing.uid}")
print(f"Name: {thing.name}")
print(f"Description: {thing.description}")

# Monitoring site information
print(f"Sampling feature type: {thing.sampling_feature_type}")
print(f"Sampling feature code: {thing.sampling_feature_code}")
print(f"Site type: {thing.site_type}")

# Location
print(f"Latitude: {thing.latitude}")
print(f"Longitude: {thing.longitude}")
print(f"Elevation: {thing.elevation_m}")
print(f"Elevation datum: {thing.elevation_datum}")

# Administrative location
print(f"Administrative area 1: {thing.admin_area_1}")
print(f"Administrative area 2: {thing.admin_area_2}")
print(f"Country: {thing.country}")

# Data information
print(f"Data disclaimer: {thing.data_disclaimer}")

# Workspace
print(f"Workspace ID: {thing.workspace_id}")
print(f"Workspace name: {thing.workspace.name}")

# Visibility
print(f"Private: {thing.is_private}")

# Tags
print(f"Tags: {thing.tags}")

# Datastreams
print(f"Datastreams: {thing.datastreams}")

### **Query Datastreams from a Monitoring Site (Thing)**

We can also query information about **Datastreams** in the database. You can return all the Datastreams you own, regardless of the monitoring site, or you can limit the list of Datastreams to a particular **monitoring site (Thing)**. The code below demonstrates both approaches. The monitoring site used here is the one returned in the previous step.


In [ ]:
# Get the Datastreams for the selected monitoring site (Thing)
thing_datastreams = thing.datastreams

# Loop through the Datastreams associated with the monitoring site
for datastream in thing_datastreams:
    print(f"Datastream ID: {datastream.uid}")
    print(f"Datastream name: {datastream.name}")

### **Query a Datastream with a Specific Name**

To retrieve a specific **Datastream** by name, you can search through the list of datastreams associated with a **Monitoring Site (Thing)**.

The code below finds the Datastream by its **name** and gets its **UUID**.

**IMPORTANT:** To search for a different Datastream, change the Datastream name in the code below.

In [ ]:
# Name of the datastream
datastream_name = "Stage - Real-time - Kanzenze Hydrological Station"

# Get the datastream by name
ds = next(
    ds for ds in thing.datastreams
    if ds.name == datastream_name
)

# Get its identifier
ds_id = ds.uid

print("Datastream ID:", ds_id)
print("Datastream name:", ds.name)

### **Query Attributes of a Datastream**

The metadata you get back from HydroServer is organized in Python objects that can easily be queried. The following code shows how to extract attributes from the **Datastream object** you just retrieved in the previous step. If you know the **UUID of a datastream**, you can use it to retrieve its metadata directly from HydroServer.

In [ ]:
# Retrieve the datastream metadata
print(f"\nRetrieving information for datastream {ds_id}\n")

print(f"Datastream name: {ds.name}")
print(f"Observed property name: {ds.observed_property.name}")
print(f"Processing level: {ds.processing_level.definition}")
print(f"Description: {ds.description}")
print(f"Unit: {ds.unit.name}")
print(f"Unit abbreviation: {ds.unit.symbol}")

### **Additional Examples of Querying Datastream Metadata**

The code below shows how to retrieve different **attributes** containing the metadata of the **datastream**.

In [ ]:
# Basic information
print(f"UID: {ds.uid}")
print(f"Name: {ds.name}")
print(f"Description: {ds.description}")

# Observed property
print(f"Observed property name: {ds.observed_property.name}")
print(f"Observed property definition: {ds.observed_property.definition}")
print(f"Observed property description: {ds.observed_property.description}")
print(f"Observed property code: {ds.observed_property.code}")

# Processing level
print(f"Processing level code: {ds.processing_level.code}")
print(f"Processing level definition: {ds.processing_level.definition}")
print(f"Processing level explanation: {ds.processing_level.explanation}")

# Unit
print(f"Unit name: {ds.unit.name}")
print(f"Unit symbol: {ds.unit.symbol}")
print(f"Unit definition: {ds.unit.definition}")
print(f"Unit type: {ds.unit.unit_type}")

# Sensor
print(f"Sensor name: {ds.sensor.name}")
print(f"Sensor description: {ds.sensor.description}")

# Monitoring site
print(f"Monitoring site: {ds.thing.name}")

# Observation metadata
print(f"Observation type: {ds.observation_type}")
print(f"Sampled medium: {ds.sampled_medium}")
print(f"No-data value: {ds.no_data_value}")
print(f"Aggregation statistic: {ds.aggregation_statistic}")

# Time information
print(f"Time aggregation interval: {ds.time_aggregation_interval}")
print(f"Time aggregation interval unit: {ds.time_aggregation_interval_unit}")
print(f"Intended time spacing: {ds.intended_time_spacing}")
print(f"Intended time spacing unit: {ds.intended_time_spacing_unit}")

# Status
print(f"Status: {ds.status}")
print(f"Result type: {ds.result_type}")
print(f"Value count: {ds.value_count}")

# Observation period
print(f"Phenomenon begin time: {ds.phenomenon_begin_time}")
print(f"Phenomenon end time: {ds.phenomenon_end_time}")

# Result period
print(f"Result begin time: {ds.result_begin_time}")
print(f"Result end time: {ds.result_end_time}")

# Visibility
print(f"Private: {ds.is_private}")
print(f"Visible: {ds.is_visible}")

## **3. Querying Observation Values from a Datastream**


---

Now that we have the UUID for the **Datastream**, we can retrieve the **Observation values** using `get_observations()`.

In the `get_observations()` function call, the `fetch_all=True` argument retrieves all **Observations** for the Datastream. Otherwise, the response is limited to **100,000 Observations**.

You can also retrieve **Observations within a specific time range** by providing a start and end date for the query.

In [ ]:
# This function call will be limited to 100,000 Observations
# obs_df = ds.get_observations().dataframe

# This function call will retrieve all Observations for the Datastream
# obs_df = datastream.get_observations(fetch_all=True).dataframe

# This function call will get the Observations between two timestamps
obs_df = ds.get_observations(
    phenomenon_time_min=datetime(year=2025, month=12, day=31),
    phenomenon_time_max=datetime(year=2026, month=8, day=13)
).dataframe

print(f'\nRetrieved {len(obs_df)} Observations for {ds.name} datastream')

### **Make a Plot of the Obervations**

We've got the Observation values, now we can generate a quick plot using Matplotlib.

In the Pandas DataFrame that is returned, the timestamps are stored in a column called "phenomenon_time", and the Observation values are stored in a column called "result". Those are the columns we need for the plot.


In [ ]:
# Make a qick plot of the Datastream using matplotlib
plt.plot(obs_df['phenomenon_time'], obs_df['result'], linestyle='-')
plt.xlabel('Date')
# Use the observed property name and unit from the Datastream metadata to label the y-axis
plt.ylabel(f'{ds.observed_property.name} ({ds.unit.symbol})')
plt.tight_layout()
plt.show()

## **What You Have Learned**

In this exercise, you learned how to use **hydroserverpy** to:

- Connect to a **HydroServer** instance using Python.
- Query **Workspaces** you have access to and those you are affiliated with.
- Find a specific **Workspace** by name and retrieve its UUID.
- Query **Monitoring Sites (Things)** within a Workspace.
- Find a specific **Monitoring Site** and query its metadata.
- Query the **Datastreams** associated with a Monitoring Site.
- Find a specific **Datastream** by name and retrieve its UUID and metadata.
- Retrieve **Observation values** from a Datastream, including observations within a specific time range.
- Work with the retrieved observations as a **Pandas DataFrame**.
- Visualize the observations using **Matplotlib**.

You now know the basic workflow for programmatically querying **HydroServer resources, metadata, and observations using Python**.

### **Additional Examples**

The code below retrieves all monitoring sites (Things) available in HydroServer, including those in public Workspaces.

In [ ]:
## Get All Things Available in HydroServer (Including Public Workspaces)
things = hs.things.list()

# Loop through all Datastreams
for thing in things.items:
    print(thing.name)

The code below retrieves all monitoring sites (Things) in the Workspaces you are associated with.

In [ ]:
# Get all Workspaces you are associated with
workspaces = hs.workspaces.list(is_associated=True).items

# Get all Things from those Workspaces
things = []

for workspace in workspaces:
    workspace_things = hs.things.list(workspace=workspace.uid).items
    things.extend(workspace_things)

# Display the Things
for thing in things:
    print(thing.name, thing.uid)

The code below retrieve all Datastreams you own, regardless of the monitoring site or workspace

In [ ]:
## Get all Datastreams you own
datastreams = hs.datastreams.list()

# Loop through all Datastreams
for datastream in datastreams.items:
    print(datastream.name)